# G10 - what the hub text member does, and is the target leaking?

**Q1** varies the hub text member across bge, SBERT, BERT, GPT-2, all four, and none. E.6 implies it should barely matter (text reaches only 2-5% of the hub) - pre-registered at under 2 points.

**Q2** separates a confound nobody has tested: in the published protocol bge is BOTH a hub member and the head target, so the hub is built with access to the directions the head must predict. Dropping it from the hub while keeping it as target isolates that, with a non-matching pair as control.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G10 — what does the hub's text member do, and is the target leaking?
# Requires G0-exact to have passed.
#
# TWO QUESTIONS, one of which should have been asked earlier.
#
# QUESTION 1 - does it matter WHICH text space joins the hub?
# The published hub is four spaces: DINOv2 small/base/large plus bge-m3.
# The project has three other text spaces cached - BERT, SBERT and GPT-2 -
# and none of them has ever been a hub member. E.6 measured how much of
# the shared coordinate system each encoder can reach: the image encoders
# reach 0.46 to 0.57, every text space reaches 0.02 to 0.05. If the hub is
# that image-dominated, swapping its text member should barely move
# transfer at all.
#
#   PRE-REGISTERED: transfer varies by less than 2 points across the text
#   variants. If it varies by more, E.6's reading is wrong and the text
#   member is doing something the reach measurement did not detect.
#
# TWO HUB CONFIGURATIONS EXIST, and the sweep covers both.
# G1_shared_hub.ipynb builds its concat from SEVEN spaces - three image
# encoders plus txt_bge, txt_gpt2, txt_bert and txt_sbert. G4_cross_lineage
# builds FOUR - the three image encoders plus txt_bge. The published 94.2
# per cent belongs to the four-space version. Both are included below as
# named variants so the answer is not accidentally specific to one.
#
# QUESTION 2 - is the head's target leaking into the hub?
# This is the one that should have been asked when the protocol was first
# written. In the published construction bge-m3 plays TWO roles at once:
# it sits inside the concatenation that defines the hub basis, AND it is
# the head's prediction target. So the hub is built with direct access to
# the very directions the head is later asked to produce.
#
# That is not obviously wrong - the hub is fitted on training rows only,
# and the head is scored on held-out rows - but it has never been
# separated, and it could inflate the 94.2 per cent. Dropping bge from the
# hub while keeping it as the target isolates it. If transfer barely
# moves, the roles are independent and the published number stands as
# measured. If it drops, the report needs a sentence.
#
#   PRE-REGISTERED: dropping the target from the hub costs less than 2
#   points. Above that, the overlap was load-bearing and must be declared.
#
# NOTE ON WHAT IS AND IS NOT A BUG. The overlap is not train/test leakage:
# the hub is fitted on training rows and every number is scored on held-out
# rows. It is an untested DEPENDENCY - the basis was constructed with
# access to the directions the head is later asked to produce, and no one
# has measured what that is worth. The original notebooks are deliberately
# left unmodified; they produced the published numbers and editing them
# would break the report's provenance. Measure it, declare it, move on.
#
# WHAT THIS IS NOT. Every variant below changes the protocol, so none of
# these numbers is comparable to the published 93.8-96.5 band. Each
# variant is scored against ITS OWN within-family baseline, which is the
# only comparison that means anything across a changing hub.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
HUB_DIM, ALPHA, N_EVAL, SEED = 512, 1e-2, 1000, 0

In [ ]:
# ---------- caches ----------
IMG = {}
for size in ("small", "base", "large"):
    IMG[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in IMG.values())
IMG = {k: v[:N] for k, v in IMG.items()}

TXT = {}
TXT["bge"] = np.load(str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]
for tag, fn, key in [("sbert", "e13_txt_sbert.npz", "txt"),
                     ("bert", "e13_txt_bert.npz", "txt"),
                     ("gpt2", "crossmodal_pairs_gpt2.npz", "txt")]:
    f = DATA_DIR / fn
    if f.exists():
        TXT[tag] = np.load(str(f))[key].astype(np.float64)[:N]
    else:
        print(f"  {tag}: {fn} missing, dropped from the sweep")

HELD = {}
for lab, fn in [("SigLIP", "e1_img_ckpt_siglip2-base-patch16-224_native.npz"),
                ("ConvNeXt", "e1_img_ckpt_convnext-base-224-22k_native.npz")]:
    f = DATA_DIR / fn
    if f.exists():
        HELD[lab] = np.load(str(f))["img"].astype(np.float64)[:N]

print(f"N = {N}")
for k, v in {**IMG, **TXT}.items():
    print(f"  {k:8s} {v.shape[1]:5d}-d   mean row norm {np.linalg.norm(v, axis=1).mean():8.2f}")
print(f"  held out: {', '.join(HELD)}")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


def build(members):
    """members: dict name -> array. Returns hub train coords + entry maps."""
    ref = np.hstack([(members[k][tr] - members[k][tr].mean(0)) /
                     (members[k][tr].std(0).mean() + 1e-12) for k in members])
    mu = ref.mean(0)
    _, sv, VT = np.linalg.svd(ref - mu, full_matrices=False)
    d = min(HUB_DIM, len(sv))
    basis = VT[:d].T / (sv[:d] / np.sqrt(len(ref)))
    return (ref - mu) @ basis, sv


def evaluate(text_member, target, source="img_small"):
    members = dict(IMG)
    if text_member == "all":
        members.update(TXT)                       # the G1 seven-space roster
    elif text_member is not None:
        members[text_member] = TXT[text_member]   # G4-style four-space

    H, sv = build(members)
    T = TXT[target]
    gal = l2n(T[te])
    to_hub = {k: ridge(IMG[k][tr], H) for k in IMG}
    head = ridge(IMG[source][tr] @ to_hub[source], T[tr])

    pcts = {}
    for enc, X in list(IMG.items()) + list(HELD.items()):
        if enc == source:
            continue
        W = to_hub[enc] if enc in to_hub else ridge(X[tr], H)
        zero = r1((X[te] @ W) @ head, gal)
        nat = r1(X[te] @ ridge(X[tr], T[tr]), gal)
        pcts[enc] = zero / max(nat, 1e-9)
    return float(np.mean(list(pcts.values()))), pcts, sv

In [ ]:
# ---------- question 1: which text member ----------
# "bge" alone IS G4's configuration; "all" IS G1's.
VARIANTS = [None] + [k for k in TXT] + ["all"]
print("\n" + "=" * 78)
print("Q1  hub text member varied, head target held at bge")
print("=" * 78)
print(f"{'hub text member':20s}{'mean % native':>15s}   per-encoder")
q1 = {}
for v in VARIANTS:
    m, per, _ = evaluate(v, "bge")
    q1[v] = m
    lbl = ("none (images only)" if v is None else
           "bge  [= G4 hub]" if v == "bge" else
           "all four  [= G1 hub]" if v == "all" else v)
    print(f"{lbl:20s}{m:>14.1%}   "
          + "  ".join(f"{k} {p:.1%}" for k, p in per.items()))

spread1 = max(q1.values()) - min(q1.values())
print(f"\nspread across text members: {spread1:.1%}")
if spread1 < 0.02:
    print("PREDICTION HELD. The hub's text member barely matters, which is")
    print("what E.6's reach measurement implies: the shared coordinate")
    print("system is very largely an image-side object, and a text space")
    print("that can only reach 2-5 per cent of it cannot shape it much.")
else:
    print("PREDICTION FAILED. The text member moves transfer by more than 2")
    print("points, so it is doing something E.6's reach measurement did not")
    print("capture. Report the spread and do not repeat the image-dominated")
    print("claim without qualifying it.")

In [ ]:
# ---------- question 2: the target-in-hub overlap ----------
print("\n" + "=" * 78)
print("Q2  is the head's target leaking into the hub?")
print("=" * 78)
print(f"{'hub contains':22s}{'target':10s}{'mean % native':>15s}")
# Full 2x2 on membership crossed with target, plus baselines. Both
# NON-matching cells are needed: with only one, an underperforming
# non-match cannot be told apart from that text space simply being a
# weaker hub member. Membership and target are independent choices -
# a space can be the head's target without being in the hub at all.
pairs = [("bge", "bge"), (None, "bge"), ("all", "bge")]
if "sbert" in TXT:
    pairs += [("sbert", "sbert"), (None, "sbert"),
              ("bge", "sbert"),      # non-matching, direction 1
              ("sbert", "bge")]      # non-matching, direction 2
q2 = {}
for member, target in pairs:
    m, _, _ = evaluate(member, target)
    q2[(member, target)] = m
    print(f"{(member or 'images only'):22s}{target:10s}{m:>14.1%}")

leak_bge = q2[("bge", "bge")] - q2[(None, "bge")]
print(f"\nbge in hub vs not, target bge: {leak_bge:+.1%}   [G4 configuration]")
print(f"seven-space hub, target bge:   "
      f"{q2[('all', 'bge')] - q2[(None, 'bge')]:+.1%}   [G1 configuration]")
if "sbert" in TXT:
    leak_sb = q2[("sbert", "sbert")] - q2[(None, "sbert")]
    x_bge_sb = q2[("bge", "sbert")] - q2[(None, "sbert")]
    x_sb_bge = q2[("sbert", "bge")] - q2[(None, "bge")]
    print(f"sbert in hub vs not, target sbert: {leak_sb:+.1%}   [matching]")
    print(f"bge in hub, target sbert:          {x_bge_sb:+.1%}   [non-matching]")
    print(f"sbert in hub, target bge:          {x_sb_bge:+.1%}   [non-matching]")

    match_mean = (leak_bge + leak_sb) / 2
    cross_mean = (x_bge_sb + x_sb_bge) / 2
    overlap = match_mean - cross_mean
    print(f"\n  mean MATCHING gain      {match_mean:+.1%}")
    print(f"  mean NON-MATCHING gain  {cross_mean:+.1%}")
    print(f"  difference              {overlap:+.1%}  <- the overlap effect")
    print("  Both non-matching cells are measured, so a weak text space")
    print("  cannot be mistaken for a mismatch penalty. Subtracting the")
    print("  non-matching mean removes the generic 'a text space in the hub")
    print("  helps' effect and leaves only what having the TARGET there is")
    print("  worth.")
    if abs(overlap) < 0.02:
        print("\n  OVERLAP NOT LOAD-BEARING on either pair. Text in the hub")
        print("  helps or does not, but it does not matter whether it is the")
        print("  target. The published protocol is clean on this point.")
    else:
        print(f"\n  OVERLAP IS REAL and worth {overlap:+.1%} beyond what any")
        print("  text member contributes. Declare it: the hub was built with")
        print("  access to the head's target space.")

print()
if abs(leak_bge) < 0.02:
    print("NO MEANINGFUL OVERLAP EFFECT. The target sitting inside the hub")
    print("is worth less than 2 points, so the published 94.2 per cent is")
    print("not resting on it. Worth one sentence in the report to say the")
    print("question was asked and answered.")
else:
    print("OVERLAP IS LOAD-BEARING. Having the head's target inside the hub")
    print(f"is worth {leak_bge:+.1%}. That does not invalidate the published")
    print("number - the hub is fitted on train rows and scored on held-out")
    print("ones - but it must be DECLARED: the hub was constructed with")
    print("access to the target space, and a hub built without it transfers")
    print("measurably worse. State it before someone finds it.")

print("\nEvery variant here changes the protocol, so none of these numbers")
print("is comparable to the published 93.8-96.5 band. Each is scored")
print("against its own within-family baseline, which is the only")
print("comparison that survives a changing hub.")